# Plant Disease Detection — Colab pipeline

Self-contained orchestration of the [`RUNBOOK.md`](RUNBOOK.md) pipeline on free-tier Colab (T4).
See [`COLAB.md`](COLAB.md) for prerequisites (Drive layout, `kaggle.json`, anti-idle JS snippet).

**Architecture:** training writes checkpoints to `/content/TRAINED/artifacts/checkpoints/` (fast SSD).
A background `rsync` loop pushes them to Drive every 30 s. On any disconnect, re-run cells
1–4, 10, 11, then re-run the training cell — `train.py --resume` continues from the last completed epoch.

## 1. Mount Drive and define paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

DRIVE_ROOT = '/content/drive/MyDrive/FYP'
LOCAL_ROOT = '/content/TRAINED'
REPO_URL = 'https://github.com/hamzayounis106/fyp-plant-disease-detection.git'

# Subfolders (created if missing). The repo-relative layout matches RUNBOOK.md.
for sub in ['', 'datasets', 'artifacts', 'artifacts/metadata',
            'artifacts/processed', 'artifacts/checkpoints',
            'artifacts/checkpoints/cnn', 'artifacts/checkpoints/efficientnet']:
    pathlib.Path(DRIVE_ROOT, sub).mkdir(parents=True, exist_ok=True)

print('Drive root :', DRIVE_ROOT)
print('Local root :', LOCAL_ROOT)
print('Repo URL   :', REPO_URL)
print('kaggle.json present:', os.path.isfile(f'{DRIVE_ROOT}/kaggle.json'))

## 2. Runtime sanity check

This project trains with standard PyTorch and requires a CUDA GPU runtime in Colab.
If TPU is selected, this cell stops early with a clear message because `train.py` is not implemented with `torch-xla`.

In [ ]:
import os
import torch

tpu_addr = os.environ.get('COLAB_TPU_ADDR', '')
if tpu_addr:
    raise SystemExit(
        'TPU runtime detected. This project uses standard PyTorch training (train.py) and '
        'is not implemented with torch-xla. Please switch Colab runtime to GPU (T4).'
    )

if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU detected. Switch runtime type to GPU (T4) and rerun.')

print('torch version :', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device name   :', torch.cuda.get_device_name(0))

## 3. Sync code from Drive to local SSD

First time only: clone the repo into `MyDrive/FYP/code/` (one-time, see [`COLAB.md`](COLAB.md) §0).
This cell mirrors `MyDrive/FYP/code/` -> `/content/TRAINED/` so we work on the fast disk.

In [ ]:
import os, subprocess

repo_dir = f'{DRIVE_ROOT}/code'
src = f'{repo_dir}/'

# Clone once if missing, otherwise pull latest.
if not os.path.isdir(repo_dir):
    subprocess.run(['git', 'clone', REPO_URL, repo_dir], check=True)
else:
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)

os.makedirs(LOCAL_ROOT, exist_ok=True)
subprocess.run(
    ['rsync', '-a', '--delete-excluded',
     '--exclude=.venv', '--exclude=__pycache__', '--exclude=.git',
     '--exclude=artifacts/checkpoints', '--exclude=artifacts/processed',
     src, LOCAL_ROOT + '/'],
    check=True,
)

os.chdir(LOCAL_ROOT)
print('cwd =', os.getcwd())
!ls -la | head -n 30

## 4. Install missing dependencies

Colab base image already has torch, torchvision, pandas, numpy, sklearn, opencv, Pillow,
matplotlib, seaborn, tqdm. Only the segmentation libraries and the Kaggle CLI are missing.

> ⚠️ Skip the cu128 line from [`RUNBOOK.md`](RUNBOOK.md) §0 — that was for RTX 50-series Blackwell.
> The default Colab torch wheel works on T4.

In [ ]:
!pip install -q rembg onnxruntime kaggle
!python -c "import torch, torchvision, pandas, numpy, sklearn, cv2, PIL, tqdm, rembg; print('deps ok')"

## 5. Datasets — Kaggle-only download (idempotent)

This cell always uses Kaggle and stores extracted datasets in `MyDrive/FYP/datasets/`.

Required once before running this cell:
- place `kaggle.json` at `MyDrive/FYP/kaggle.json`

The script downloads and unzips into the exact folder names expected by
`scripts/build_unified_metadata.py`:
- `Plant Village Dataset 2`
- `PlantDoc Classification dataset 1`

In [ ]:
import os, shutil, subprocess, pathlib

# Fixed config (no placeholders).
PV_DIR_NAME = 'Plant Village Dataset 2'
PD_DIR_NAME = 'PlantDoc Classification dataset 1'
PV_KAGGLE_SLUG = 'abdallahalidev/plantvillage-dataset'
PD_KAGGLE_SLUG = 'nirmalsankalana/plantdoc-dataset'

# 1) Stage kaggle.json from Drive.
home_kaggle = pathlib.Path('/root/.kaggle')
home_kaggle.mkdir(parents=True, exist_ok=True)
src = pathlib.Path(DRIVE_ROOT, 'kaggle.json')
if not src.is_file():
    raise SystemExit(
        f'Missing {src}. Download kaggle.json from Kaggle Account settings and place it in Drive root.'
    )
shutil.copy(src, home_kaggle / 'kaggle.json')
os.chmod(home_kaggle / 'kaggle.json', 0o600)

# 2) Kaggle download targets in Drive.
drive_pv = pathlib.Path(DRIVE_ROOT, 'datasets', PV_DIR_NAME)
drive_pd = pathlib.Path(DRIVE_ROOT, 'datasets', PD_DIR_NAME)

def kaggle_fetch(slug, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    if any(target_dir.iterdir()):
        print(f'[skip] {target_dir} already populated')
        return
    print(f'[kaggle] {slug} -> {target_dir}')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', slug,
                    '-p', str(target_dir), '--unzip'], check=True)

kaggle_fetch(PV_KAGGLE_SLUG, drive_pv)
kaggle_fetch(PD_KAGGLE_SLUG, drive_pd)

# 3) Symlink Drive copy into LOCAL_ROOT so RUNBOOK paths stay identical.
for name, drive_path in [(PV_DIR_NAME, drive_pv), (PD_DIR_NAME, drive_pd)]:
    local_path = pathlib.Path(LOCAL_ROOT, name)
    if local_path.is_symlink() or local_path.exists():
        if local_path.is_symlink():
            local_path.unlink()
        elif local_path.is_dir():
            shutil.rmtree(local_path)
        else:
            local_path.unlink()
    os.symlink(drive_path, local_path)
    print(f'{local_path} -> {os.readlink(local_path)}')

# 4) Quick layout sanity check.
pv_crops = sorted(p.name for p in drive_pv.iterdir() if p.is_dir())[:5]
pd_splits = sorted(p.name for p in drive_pd.iterdir() if p.is_dir())[:5]
print('PV first 5 crops :', pv_crops)
print('PD splits        :', pd_splits)

## 6. Build unified metadata

Mirrors [`RUNBOOK.md`](RUNBOOK.md) §1. Outputs `unified_metadata.csv` (~70k rows, 36 classes).

In [ ]:
%cd /content/TRAINED
!python scripts/build_unified_metadata.py \
    --plant-village-root "Plant Village Dataset 2" \
    --plantdoc-root "PlantDoc Classification dataset 1"
!ls -la artifacts/metadata/

## 7. Pre-resize 224×224 JPEG cache

[`RUNBOOK.md`](RUNBOOK.md) §2. Uses `--workers 2` because Colab CPUs are weak (6 starves).
Output goes under `artifacts/processed/resized/`. On T4, expect ~15–25 min.

In [ ]:
%cd /content/TRAINED
!python -u scripts/cache_resized.py \
    --metadata-csv artifacts/metadata/unified_metadata.csv \
    --output-csv  artifacts/metadata/unified_metadata_resized.csv \
    --size 224 --workers 2

## 8. Segment the test set

[`RUNBOOK.md`](RUNBOOK.md) §3 verbatim. ~15–20 min on Colab CPU.

In [ ]:
%cd /content/TRAINED

# 8a. Slice test rows out of the unified metadata.
!python -c "import pandas as pd; df=pd.read_csv('artifacts/metadata/unified_metadata.csv'); df[df['split']=='test'].to_csv('artifacts/metadata/unified_test_only.csv', index=False)"

# 8b. Run segmentation (grabCut, faster than rembg).
!python scripts/preprocess_segment.py \
    --metadata-csv artifacts/metadata/unified_test_only.csv \
    --output-root artifacts/processed \
    --output-metadata-csv artifacts/metadata/unified_test_segmented.csv \
    --force-grabcut

# 8c. Filter out rows where segmentation failed entirely (~13 of 1610).
!python -c "import os, pandas as pd; df=pd.read_csv('artifacts/metadata/unified_test_segmented.csv'); df=df[df['processed_path'].apply(os.path.exists)].copy(); df.to_csv('artifacts/metadata/unified_test_segmented_ok.csv', index=False); print(f'kept {len(df)} rows')"

## 9. One-shot sync of metadata + resized cache to Drive

If the runtime dies between here and the start of training, we don't want to redo cells 7–8
(~30 min). Push the artifacts to Drive once. The resized cache is large but it lives in Drive
in case we ever need it again. From here on, only the much smaller `checkpoints/` folder is
re-synced continuously (cell 10).

In [ ]:
!rsync -a --info=progress2 \
    /content/TRAINED/artifacts/metadata/ \
    /content/drive/MyDrive/FYP/artifacts/metadata/

!rsync -a --info=progress2 \
    /content/TRAINED/artifacts/processed/ \
    /content/drive/MyDrive/FYP/artifacts/processed/

!du -sh /content/drive/MyDrive/FYP/artifacts/*

## 10. Start the background checkpoint watcher

A 4-line bash background loop syncs `artifacts/checkpoints/` every 30 s. Only ~5 MB (CNN) +
~230 MB (EffNet) `last.pt` + `best.pt`, so the upload finishes in seconds and stays well
under any Drive write rate limits.

The PID is saved to `/content/sync.pid`. Kill it at the end with cell 16.

In [ ]:
%%bash
nohup bash -c '
  while true; do
    rsync -a --update       /content/TRAINED/artifacts/checkpoints/       /content/drive/MyDrive/FYP/artifacts/checkpoints/ 2>/dev/null
    sleep 30
  done
' > /content/sync.log 2>&1 &
echo $! > /content/sync.pid
echo "watcher PID: $(cat /content/sync.pid)"

## 11. Pull checkpoints from Drive (run after every reconnect)

Idempotent. Run this *before* re-launching a training cell. After it completes, `train.py --resume`
will find the right `last.pt`. On a fresh first run there's nothing to pull and it's a no-op.

In [ ]:
!rsync -a /content/drive/MyDrive/FYP/artifacts/checkpoints/ \
          /content/TRAINED/artifacts/checkpoints/
!ls -la /content/TRAINED/artifacts/checkpoints/cnn/ \
        /content/TRAINED/artifacts/checkpoints/efficientnet/ 2>/dev/null

## 12. Train SimpleCNN — 25 epochs

[`RUNBOOK.md`](RUNBOOK.md) §4. The `--resume` flag is **always present** and is a no-op on the
first run (no `last.pt` yet). On any subsequent run it picks up from the last completed epoch.
Re-running this cell after Colab finishes 25 epochs is safe — `train.py` short-circuits with
`"Already completed 25 epochs"`.

T4: ~2–3 h. The watcher syncs `last.pt` to Drive every 30 s while this runs.

In [ ]:
%cd /content/TRAINED
!python -u train.py \
    --metadata-csv artifacts/metadata/unified_metadata_resized.csv \
    --image-column image_path_resized \
    --model cnn --epochs 25 --batch-size 64 --image-size 224 \
    --amp --num-workers 2 --prefetch-factor 2 \
    --output-dir artifacts/checkpoints/cnn \
    --resume

## 13. Train EfficientNetV2-S — Stage A (20 ep) then Stage B (+5 fine-tune)

[`RUNBOOK.md`](RUNBOOK.md) §5. Two `train.py` invocations. Stage B uses
`--resume --fresh-schedule --lr 5e-5` (the cosine LR is rebuilt over the remaining 5 epochs at
the new lower lr). Both calls are idempotent re-run-safe.

T4: ~3.5–5 h total. If session dies between A and B, just re-run this cell after pulling
checkpoints (cell 11) — Stage A short-circuits, Stage B picks up.

In [ ]:
%cd /content/TRAINED

# Stage A: 20 epochs base.
!python -u train.py \
    --metadata-csv artifacts/metadata/unified_metadata_resized.csv \
    --image-column image_path_resized \
    --model efficientnet --use-pretrained --epochs 20 --batch-size 64 --image-size 224 \
    --amp --num-workers 2 --prefetch-factor 2 \
    --output-dir artifacts/checkpoints/efficientnet \
    --resume

# Stage B: +5 fine-tune at lr 5e-5.
!python -u train.py \
    --metadata-csv artifacts/metadata/unified_metadata_resized.csv \
    --image-column image_path_resized \
    --model efficientnet --use-pretrained --epochs 25 --batch-size 64 --image-size 224 \
    --amp --num-workers 2 --prefetch-factor 2 \
    --output-dir artifacts/checkpoints/efficientnet \
    --resume --fresh-schedule --lr 5e-5

## 14. Merge duplicate corn-rust class, then calibrate

[`RUNBOOK.md`](RUNBOOK.md) §§6–7. The class-merge step is only needed if you trained against the
old `plantdoc_alias.json` (37 classes); on the corrected metadata you have 36 classes and the
merge is harmless idempotent surgery (it errors out if indices don't exist — handled below).

Calibration writes `best_calibrated.pt` next to the input checkpoint and prints the chosen
strategy.

In [ ]:
%cd /content/TRAINED
import os

for model in ['cnn', 'efficientnet']:
    src_ckpt   = f'artifacts/checkpoints/{model}/best.pt'
    merged     = f'artifacts/checkpoints/{model}/best_merged.pt'
    if not os.path.isfile(src_ckpt):
        print(f'[skip] {src_ckpt} missing — train first.')
        continue

    # Try the merge. If the head already has 36 classes, this fails — fall back to copying
    # best.pt -> best_merged.pt so the calibration step finds the same input file.
    rc = os.system(
        f'python scripts/merge_class.py '
        f'--checkpoint {src_ckpt} '
        f'--out-checkpoint {merged} '
        f'--src-idx 10 --target-idx 9'
    )
    if rc != 0 or not os.path.isfile(merged):
        print(f'[merge skipped for {model} — copying best.pt -> best_merged.pt]')
        import shutil
        shutil.copy(src_ckpt, merged)

# Calibrate both.
!python scripts/logit_adjust.py --checkpoint artifacts/checkpoints/cnn/best_merged.pt
!python scripts/logit_adjust.py --checkpoint artifacts/checkpoints/efficientnet/best_merged.pt

!ls -la artifacts/checkpoints/cnn/ artifacts/checkpoints/efficientnet/

## 15. Run the 8-eval matrix

[`RUNBOOK.md`](RUNBOOK.md) §8: `{cnn, efficientnet} × {plant_village, plantdoc} × {raw, segmented}`.
Uses the bundled `scripts/run_eval_matrix.sh` (Colab is Linux, so bash works directly).

In [ ]:
%cd /content/TRAINED
!bash scripts/run_eval_matrix.sh best_calibrated.pt

## 16. Stop the background watcher and final flush to Drive

One last `rsync` so all eight `eval_*.json` files and `best_calibrated.pt` for both models
land in Drive immediately, then kill the watcher.

In [ ]:
import os, signal, pathlib

# Final one-shot sync of everything we care about.
!rsync -a /content/TRAINED/artifacts/ /content/drive/MyDrive/FYP/artifacts/

# Stop the background watcher.
pid_file = pathlib.Path('/content/sync.pid')
if pid_file.is_file():
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f'killed watcher pid {pid}')
    except ProcessLookupError:
        print(f'watcher pid {pid} already gone')
    pid_file.unlink()
else:
    print('no /content/sync.pid — watcher was not running')

## 17. Summary of all 8 evaluations

In [ ]:
%cd /content/TRAINED
!python scripts/summarize_evals.py